# 4 · Day 2 – Silver & Data Quality
Input: the 6 Bronze tables from Day 1 in `capstone_5iot.iot_schema` (Final_dataset).
Output: clean, typed tables in the new schema **`capstone_5iot.silver`**.

| Task | What this notebook does | Output tables |
|---|---|---|
| 1 | DQ rule catalogue | `dq_rule_catalogue` |
| 2 | Cast numbers / timestamps, keep raw record for traceability | `silver_*` |
| 3 | Quarantine invalid rows with rule ID + reason + source file | `silver_rejected_records` |
| 4 | Deduplicate sensor readings (deterministic) + audit counts | `silver_load_audit` |
| 5 | Status CDC with MERGE (late / out-of-order / replay safe) | `silver_work_order_status_history`, `silver_work_order_current` |
| 6 | Domain logic: sensor windows, maintenance episodes, missing telemetry | `silver_sensor_windows`, `silver_maintenance_episodes`, `silver_telemetry_gaps` |
| 7 | Unit tests + final validation | – |

Every check is collected with `check()` and shown in one table at the end (section 8).

## 0. Setup

In [0]:
#hardcode for pipeline day 4 task 2
# dbutils.widgets.text("catalog", "capstone_5iot")
# dbutils.widgets.text("bronze_schema", "iot_schema")
# dbutils.widgets.text("silver_schema", "silver")
# CATALOG = dbutils.widgets.get("catalog")
# BRONZE  = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}"
# SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"




from pyspark.sql import functions as F, Window
#org
BRONZE = "capstone_5iot.iot_schema"     # Day 1 Bronze tables (read only)
SILVER = "capstone_5iot.silver"         # all Day 2 tables are written here

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"USE {SILVER}")              # table names without a schema now point to silver

results = []   # all validation results

def check(task, name, actual, expected):
    status = "PASS" if actual == expected else "FAIL"
    results.append((task, name, str(actual), str(expected), status))
    print(f"[{status}] {task} | {name}: actual={actual} expected={expected}")

def save(df, table):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

def num(c):  return F.expr(f"try_cast({c} AS DOUBLE)").alias(c)
def ts(c):   return F.expr(f"try_cast({c} AS TIMESTAMP)").alias(c)

def raw_record(df):
    """All source columns as one JSON string = the original raw values."""
    return F.to_json(F.struct(*[c for c in df.columns if not c.startswith("_")])).alias("_raw_record")

## 1. DQ rule catalogue
REJECT = goes to `silver_rejected_records` · DEDUPLICATE = only first copy kept · FLAG = row kept with a flag column · KEEP = allowed · IGNORE = change not applied

In [0]:
rules = [
    ("DQ01", "sensor_readings", "reading_id",    "Business key must not be null",                         "CRITICAL", "REJECT"),
    ("DQ02", "sensor_readings", "reading_ts",    "Must be a valid timestamp",                             "CRITICAL", "REJECT"),
    ("DQ03", "sensor_readings", "machine_id",    "Must exist in machine master",                          "CRITICAL", "REJECT"),
    ("DQ04", "sensor_readings", "temperature_c", "Must be between -50 and 200 (999999.99 / -999 = sensor error)", "HIGH", "REJECT"),
    ("DQ05", "sensor_readings", "reading_id",    "Duplicate reading - keep first ingested copy",          "MEDIUM",   "DEDUPLICATE"),
    ("DQ06", "sensor_readings", "pressure_bar, operating_mode, sensor_quality, firmware_version",
                                                 "Optional fields may be null",                           "LOW",      "KEEP"),
    ("DQ07", "sensor_readings", "reading_ts",    "Reading before machine commissioned_date",              "LOW",      "FLAG"),
    ("DQ08", "sensor_readings", "machine_id",    "Machine operating_status is INACTIVE",                  "LOW",      "FLAG"),
    ("DQ09", "failure_events / work_orders", "machine_id, work_order_id, failure_code",
                                                 "Keys must exist in their master / parent table",        "CRITICAL", "REJECT"),
    ("DQ10", "failure_events / work_orders", "failure_ts, created_ts",
                                                 "Must be a valid timestamp",                             "CRITICAL", "REJECT"),
    ("DQ11", "work_orders",     "completed_ts",  "completed_ts filled but status is not COMPLETED",       "MEDIUM",   "FLAG"),
    ("DQ12", "work_order_status_cdc", "sequence_no", "Older or repeated status change must not overwrite newer", "HIGH", "IGNORE"),
]
save(spark.createDataFrame(rules, "rule_id string, table_name string, column_name string, rule string, severity string, action string"),
     "dq_rule_catalogue")
display(spark.table("dq_rule_catalogue"))

check("T1", "rules in catalogue", spark.table("dq_rule_catalogue").count(), 12)

## 2. Typed Silver tables
### 2a. Reference tables (from Azure SQL Bronze)

In [0]:
machines = spark.table(f"{BRONZE}.bronze_machine_master").select(
    "machine_id", "plant_id", "machine_type", "manufacturer",
    F.expr("try_cast(commissioned_date AS DATE)").alias("commissioned_date"),
    "criticality", num("rated_rpm"), "operating_status",
    F.expr("try_cast(maintenance_interval_days AS INT)").alias("maintenance_interval_days"), "line_id")

plants = spark.table(f"{BRONZE}.bronze_plant_master").select(
    "plant_id", "plant_name", "city", "region", "country", "plant_type", "active_flag",
    F.expr("try_cast(production_line_count AS INT)").alias("production_line_count"), "timezone", "plant_manager")

codes = spark.table(f"{BRONZE}.bronze_failure_code_lookup").select(
    [c for c in spark.table(f"{BRONZE}.bronze_failure_code_lookup").columns if not c.startswith("_")])

save(machines, "silver_machine_master")
save(plants,   "silver_plant_master")
save(codes,    "silver_failure_code_lookup")

check("T2", "silver_machine_master rows",      spark.table("silver_machine_master").count(), 150)
check("T2", "silver_plant_master rows",        spark.table("silver_plant_master").count(), 159)
check("T2", "silver_failure_code_lookup rows", spark.table("silver_failure_code_lookup").count(), 168)
check("T2", "commissioned_date not castable",  spark.table("silver_machine_master").filter("commissioned_date IS NULL").count(), 0)

### 2b. Sensor readings – cast → deduplicate → quality rules
Same 3 functions are reused in the unit tests (section 7).

In [0]:
def type_sensor(df):
    """Task 2: cast to proper types, keep lineage + raw record."""
    return df.select(
        "reading_id", "machine_id", "plant_id", ts("reading_ts"),
        num("temperature_c"), num("vibration_mm_s"), num("pressure_bar"), num("rpm"),
        "operating_mode", "sensor_quality", "firmware_version", ts("received_ts"),
        "_source_file", "_ingestion_ts", raw_record(df))

def dedup(df):
    """Task 4: number copies of each reading_id. Tie-breaker: earliest _ingestion_ts, then _source_file."""
    w = Window.partitionBy("reading_id").orderBy("_ingestion_ts", "_source_file")
    return df.withColumn("copy_no", F.row_number().over(w))      # 1 = keep, >1 = duplicate

def apply_sensor_rules(df):
    """Task 3: add machine info and the first broken rule (NULL = valid)."""
    m = spark.table("silver_machine_master").select(
        "machine_id", "commissioned_date", F.col("operating_status").alias("machine_status"),
        F.lit(True).alias("machine_exists"))
    return (df.join(m, "machine_id", "left")
              .withColumn("rule_id",
                  F.when(F.col("copy_no") > 1,                       "DQ05")
                   .when(F.col("reading_id").isNull(),               "DQ01")
                   .when(F.col("reading_ts").isNull(),               "DQ02")
                   .when(F.col("machine_exists").isNull(),           "DQ03")
                   .when(~F.col("temperature_c").between(-50, 200),  "DQ04")))

In [0]:
bronze_sensor  = spark.table(f"{BRONZE}.bronze_sensor_readings")
checked_sensor = apply_sensor_rules(dedup(type_sensor(bronze_sensor)))

ABNORMAL = (F.col("temperature_c") > 85) | (F.col("vibration_mm_s") > 7.1) | (F.col("pressure_bar") > 18)

silver_sensor = (checked_sensor.filter("rule_id IS NULL")
    .withColumn("is_before_commissioning", F.to_date("reading_ts") < F.col("commissioned_date"))    # DQ07
    .withColumn("is_inactive_machine",     F.col("machine_status") == "INACTIVE")                    # DQ08
    .withColumn("has_missing_optional",    F.col("pressure_bar").isNull() | F.col("operating_mode").isNull()
                                           | F.col("sensor_quality").isNull() | F.col("firmware_version").isNull())
    .withColumn("is_abnormal",             F.coalesce(ABNORMAL, F.lit(False)))
    .drop("rule_id", "copy_no", "machine_exists", "commissioned_date", "machine_status", "_raw_record"))

save(silver_sensor, "silver_sensor_readings")

### 2c. Failure events and work orders

In [0]:
def exists(df, col, table, flag):
    """Left-join a key list from another table -> flag column is True when the key exists."""
    keys = spark.table(table).select(col).distinct().withColumn(flag, F.lit(True))
    return df.join(keys, col, "left")

# ---- failure events ----
bronze_fe = spark.table(f"{BRONZE}.bronze_failure_events")
fe = bronze_fe.select(
    "failure_event_id", "machine_id", ts("failure_ts"), "failure_code", "severity",
    F.expr("try_cast(downtime_minutes AS INT)").alias("downtime_minutes"),
    "reported_by", "work_order_id", "plant_id", "event_status", "_source_file", "_ingestion_ts", raw_record(bronze_fe))
fe = exists(fe, "machine_id",    "silver_machine_master",      "machine_ok")
fe = exists(fe, "work_order_id", f"{BRONZE}.bronze_work_orders", "wo_ok")
fe = exists(fe, "failure_code",  "silver_failure_code_lookup", "code_ok")
checked_fe = fe.withColumn("rule_id",
    F.when(F.col("failure_ts").isNull(), "DQ10")
     .when(F.col("machine_ok").isNull() | F.col("wo_ok").isNull() | F.col("code_ok").isNull(), "DQ09"))

save(checked_fe.filter("rule_id IS NULL").drop("machine_ok", "wo_ok", "code_ok", "rule_id", "_raw_record"),
     "silver_failure_events")

# ---- work orders ----
bronze_wo = spark.table(f"{BRONZE}.bronze_work_orders")
wo = bronze_wo.select(
    "work_order_id", "machine_id", ts("created_ts"), ts("planned_start_ts"), ts("completed_ts"),
    "maintenance_type", "technician_id", "work_order_status", num("parts_cost"), "failure_code",
    "_source_file", "_ingestion_ts", raw_record(bronze_wo))
wo = exists(wo, "machine_id",   "silver_machine_master",      "machine_ok")
wo = exists(wo, "failure_code", "silver_failure_code_lookup", "code_ok")
checked_wo = wo.withColumn("rule_id",
    F.when(F.col("created_ts").isNull(), "DQ10")
     .when(F.col("machine_ok").isNull() | F.col("code_ok").isNull(), "DQ09"))

silver_wo = (checked_wo.filter("rule_id IS NULL")
    .withColumn("is_completed_ts_invalid",                                                          # DQ11
                (F.col("work_order_status") != "COMPLETED") & F.col("completed_ts").isNotNull())
    .withColumn("completed_ts_clean",
                F.when(F.col("work_order_status") == "COMPLETED", F.col("completed_ts")))
    .drop("machine_ok", "code_ok", "rule_id", "_raw_record"))
save(silver_wo, "silver_work_orders")

## 3. Quarantine – `silver_rejected_records`
One row per rejected record: table, key, rule ID, reason, source file and the full raw record.

In [0]:
reasons = spark.table("dq_rule_catalogue").select("rule_id", F.col("rule").alias("reason"))

def rejected(df, table, key):
    return (df.filter("rule_id IS NOT NULL")
              .join(reasons, "rule_id")
              .select(F.lit(table).alias("source_table"), F.col(key).alias("record_key"),
                      "rule_id", "reason", "_source_file", "_raw_record",
                      F.current_timestamp().alias("rejected_ts")))

rejected_all = (rejected(checked_sensor, "bronze_sensor_readings", "reading_id")
    .unionByName(rejected(checked_fe,    "bronze_failure_events",  "failure_event_id"))
    .unionByName(rejected(checked_wo,    "bronze_work_orders",     "work_order_id")))
save(rejected_all, "silver_rejected_records")

display(spark.sql("""SELECT source_table, rule_id, reason, COUNT(*) AS records
                     FROM silver_rejected_records GROUP BY ALL ORDER BY rule_id"""))

In [0]:
rej = spark.table("silver_rejected_records")
check("T3", "sensor rejected DQ04 (temperature sentinels)", rej.filter("rule_id = 'DQ04'").count(), 46)
check("T3", "sensor rejected DQ01/DQ02/DQ03",               rej.filter("rule_id IN ('DQ01','DQ02','DQ03')").count(), 0)
check("T3", "failure events rejected",                      rej.filter("source_table = 'bronze_failure_events'").count(), 0)
check("T3", "work orders rejected",                         rej.filter("source_table = 'bronze_work_orders'").count(), 0)
check("T3", "every rejected row has source file + raw record",
      rej.filter("_source_file IS NULL OR _raw_record IS NULL").count(), 0)

## 4. Deduplication & before/after audit – `silver_load_audit`
Reconciliation rule: **bronze rows = silver rows + duplicates removed + rejected**

In [0]:
def audit(bronze, silver):
    b = spark.table(f"{BRONZE}.{bronze}").count()
    d = rej.filter(f"source_table = '{bronze}' AND rule_id = 'DQ05'").count()
    r = rej.filter(f"source_table = '{bronze}' AND rule_id <> 'DQ05'").count()
    s = spark.table(silver).count()
    return (bronze, silver, b, d, r, s, b == s + d + r)

audit_rows = [audit("bronze_sensor_readings", "silver_sensor_readings"),
              audit("bronze_failure_events",  "silver_failure_events"),
              audit("bronze_work_orders",     "silver_work_orders")]
save(spark.createDataFrame(audit_rows, "bronze_table string, silver_table string, bronze_rows long, "
                                       "duplicates_removed long, rejected long, silver_rows long, reconciled boolean")
          .withColumn("audit_ts", F.current_timestamp()), "silver_load_audit")
display(spark.table("silver_load_audit"))

s = spark.table("silver_sensor_readings")
check("T4", "sensor duplicates removed",          rej.filter("rule_id = 'DQ05'").count(), 41)
check("T4", "silver_sensor_readings rows",        s.count(), 11566)
check("T4", "reading_id unique in Silver",        s.count() - s.select("reading_id").distinct().count(), 0)
check("T4", "all tables reconciled",              spark.table("silver_load_audit").filter("NOT reconciled").count(), 0)
check("T4", "silver_failure_events rows",         spark.table("silver_failure_events").count(), 665)
check("T4", "silver_work_orders rows",            spark.table("silver_work_orders").count(), 335)
check("T2", "reading_ts typed as timestamp",      dict(s.dtypes)["reading_ts"], "timestamp")
check("T2", "flag DQ07 readings before commissioning", s.filter("is_before_commissioning").count(), 2045)
check("T2", "flag DQ08 readings on INACTIVE machines", s.filter("is_inactive_machine").count(), 3382)
check("T2", "flag DQ06 rows with missing optional (15 values in 14 rows)", s.filter("has_missing_optional").count(), 14)
check("T2", "flag DQ11 work orders completed_ts invalid (none: completed_ts only on COMPLETED)",
      spark.table("silver_work_orders").filter("is_completed_ts_invalid").count(), 0)

## 5. Status CDC – work-order status changes
The Final_dataset has **no status-change feed**, so a CDC feed is **simulated** from `maintenance_work_orders.csv`
(documented limitation): OPEN at `created_ts` → IN_PROGRESS at `planned_start_ts` → COMPLETED at `completed_ts`,
following each order's final `work_order_status`. `sequence_no` gives the order of changes.

- **Batch 1** – OPEN + IN_PROGRESS changes arrive on time
- **Batch 2** – COMPLETED changes arrive **late**, plus 20 **old OPEN rows replayed** (out-of-order / unchanged replay)
- **Batch 2 again** – full replay, nothing may change

`silver_work_order_status_history` keeps every valid change once. `silver_work_order_current` is only updated when the incoming `sequence_no` is **newer** (rule DQ12).

In [0]:
wo_s = spark.table("silver_work_orders")
cdc = (wo_s.select("work_order_id", F.lit(1).alias("sequence_no"), F.lit("OPEN").alias("status"),
                   F.col("created_ts").alias("effective_ts"))
       .unionByName(wo_s.filter("work_order_status IN ('IN_PROGRESS', 'COMPLETED')")
                        .select("work_order_id", F.lit(2), F.lit("IN_PROGRESS"), "planned_start_ts")
                        .toDF("work_order_id", "sequence_no", "status", "effective_ts"))
       .unionByName(wo_s.filter("work_order_status = 'COMPLETED'")
                        .select("work_order_id", F.lit(3), F.lit("COMPLETED"), "completed_ts")
                        .toDF("work_order_id", "sequence_no", "status", "effective_ts"))
       .withColumn("_source_system", F.lit("SIMULATED from maintenance_work_orders.csv")))
# Day 5 fix: Bronze is immutable - seed the simulated CDC feed only once, then Silver only reads it
CDC_TABLE = f"{BRONZE}.bronze_work_order_status_cdc"
if not spark.catalog.tableExists(CDC_TABLE):
    save(cdc, CDC_TABLE)          # one-time seed of the simulated source feed
cdc = spark.table(CDC_TABLE)      # from now on Silver only reads Bronze

spark.sql("DROP TABLE IF EXISTS silver_work_order_status_history")
spark.sql("DROP TABLE IF EXISTS silver_work_order_current")
spark.sql("CREATE TABLE silver_work_order_status_history (work_order_id STRING, sequence_no INT, status STRING, effective_ts TIMESTAMP, processed_ts TIMESTAMP)")
spark.sql("CREATE TABLE silver_work_order_current (work_order_id STRING, sequence_no INT, status STRING, effective_ts TIMESTAMP, updated_ts TIMESTAMP)")

In [0]:
def process_cdc(batch):
    batch = batch.dropDuplicates(["work_order_id", "sequence_no"])

    # 1. history: insert only changes not seen before (replays are ignored)
    batch.createOrReplaceTempView("cdc_batch")
    spark.sql("""
        MERGE INTO silver_work_order_status_history t
        USING cdc_batch s
        ON t.work_order_id = s.work_order_id AND t.sequence_no = s.sequence_no
        WHEN NOT MATCHED THEN INSERT (work_order_id, sequence_no, status, effective_ts, processed_ts)
                              VALUES (s.work_order_id, s.sequence_no, s.status, s.effective_ts, current_timestamp())
    """)

    # 2. current: latest change per work order, applied only if newer than what we have
    w = Window.partitionBy("work_order_id").orderBy(F.desc("sequence_no"))
    batch.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn").createOrReplaceTempView("cdc_latest")
    spark.sql("""
        MERGE INTO silver_work_order_current t
        USING cdc_latest s
        ON t.work_order_id = s.work_order_id
        WHEN MATCHED AND s.sequence_no > t.sequence_no THEN
             UPDATE SET sequence_no = s.sequence_no, status = s.status, effective_ts = s.effective_ts, updated_ts = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (work_order_id, sequence_no, status, effective_ts, updated_ts)
                              VALUES (s.work_order_id, s.sequence_no, s.status, s.effective_ts, current_timestamp())
    """)

def cdc_counts():
    return (spark.table("silver_work_order_status_history").count(),
            spark.table("silver_work_order_current").count())

In [0]:
batch1 = cdc.filter("sequence_no < 3")
batch2 = (cdc.filter("sequence_no = 3")                                         # late COMPLETED changes
             .unionByName(cdc.filter("sequence_no = 1").orderBy("work_order_id").limit(20)))   # old OPEN replayed

process_cdc(batch1)
after_b1 = cdc_counts()
process_cdc(batch2)
after_b2 = cdc_counts()
process_cdc(batch2)                                                             # replay the same batch
after_replay = cdc_counts()
print("history / current rows  batch1:", after_b1, " batch2:", after_b2, " replay:", after_replay)

cur = spark.table("silver_work_order_current")
wrong_status = cur.join(wo_s, "work_order_id").filter("status <> work_order_status").count()
open_overwrite = (cur.join(cdc.filter("sequence_no = 1").orderBy("work_order_id").limit(20).select("work_order_id"), "work_order_id")
                     .join(wo_s, "work_order_id").filter("work_order_status <> 'OPEN' AND status = 'OPEN'").count())

check("T5", "CDC events generated",                         cdc.count(), 652)
check("T5", "history rows after all batches",               after_b2[0], 652)
check("T5", "current rows = work orders (1 per key)",       after_b2[1], 335)
check("T5", "late COMPLETED applied (completed orders)",    cur.filter("status = 'COMPLETED'").count(), 106)
check("T5", "out-of-order OPEN did not overwrite newer",    open_overwrite, 0)
check("T5", "current status = source status",               wrong_status, 0)
check("T5", "replay changed nothing",                       after_replay, after_b2)
check("T5", "history: no duplicate (work_order_id, sequence_no)",
      spark.table("silver_work_order_status_history").groupBy("work_order_id", "sequence_no").count().filter("count > 1").count(), 0)

In [0]:
%sql
-- full status history of one completed work order
SELECT * FROM silver_work_order_status_history
WHERE work_order_id = (SELECT MIN(work_order_id) FROM silver_work_order_current WHERE status = 'COMPLETED')
ORDER BY sequence_no

## 6. Domain logic
**Abnormal reading** (assumption, agree with client): temperature > 85 °C **or** vibration > 7.1 mm/s **or** pressure > 18 bar.

### 6a. Aggregate sensor windows – `silver_sensor_windows` (1-day windows per machine)

In [0]:
WINDOW = "1 day"

windows = (spark.table("silver_sensor_readings")
    .groupBy("machine_id", "plant_id", F.window("reading_ts", WINDOW).alias("w"))
    .agg(F.count("*").alias("reading_count"),
         F.sum(F.col("is_abnormal").cast("int")).alias("abnormal_count"),
         F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
         F.round(F.max("vibration_mm_s"), 2).alias("max_vibration_mm_s"),
         F.round(F.avg("pressure_bar"), 2).alias("avg_pressure_bar"),
         F.round(F.avg("rpm"), 2).alias("avg_rpm"))
    .select("machine_id", "plant_id", F.col("w.start").alias("window_start"), F.col("w.end").alias("window_end"),
            "reading_count", "abnormal_count", "avg_temperature_c", "max_vibration_mm_s", "avg_pressure_bar", "avg_rpm"))
save(windows, "silver_sensor_windows")

sw = spark.table("silver_sensor_windows")
check("T6", "window readings = silver readings", sw.agg(F.sum("reading_count")).first()[0], 11566)
check("T6", "one row per machine per window",    sw.count() - sw.select("machine_id", "window_start").distinct().count(), 0)

### 6b. Join maintenance episodes – `silver_maintenance_episodes`
One row per failure: linked work order + sensor readings in the 24 h before the failure.

In [0]:
LOOKBACK_HOURS = 24

failures = spark.table("silver_failure_events")
readings = spark.table("silver_sensor_readings").select("machine_id", "reading_ts", "vibration_mm_s", "is_abnormal")

before_failure = (failures.select("failure_event_id", "machine_id", "failure_ts")
    .join(readings, "machine_id")
    .filter((F.col("reading_ts") <= F.col("failure_ts")) &
            (F.col("reading_ts") >  F.col("failure_ts") - F.expr(f"INTERVAL {LOOKBACK_HOURS} HOURS")))
    .groupBy("failure_event_id")
    .agg(F.count("*").alias("readings_before_failure"),
         F.sum(F.col("is_abnormal").cast("int")).alias("abnormal_before_failure"),
         F.round(F.avg("vibration_mm_s"), 2).alias("avg_vibration_before_failure")))

episodes = (failures.select("failure_event_id", "machine_id", "plant_id", "failure_ts", "failure_code",
                            "severity", "downtime_minutes", "event_status", "work_order_id")
    .join(spark.table("silver_work_orders").select("work_order_id", "maintenance_type", "work_order_status",
                                                   "completed_ts_clean", "parts_cost"), "work_order_id", "left")
    .join(before_failure, "failure_event_id", "left")
    .fillna(0, ["readings_before_failure", "abnormal_before_failure"])
    .withColumn("repair_hours",
                F.round((F.unix_timestamp("completed_ts_clean") - F.unix_timestamp("failure_ts")) / 3600, 2)))
save(episodes, "silver_maintenance_episodes")

ep = spark.table("silver_maintenance_episodes")
check("T6", "episodes = failure events",          ep.count(), 665)
check("T6", "episodes without work order",        ep.filter("maintenance_type IS NULL").count(), 0)
check("T6", "episodes with readings in last 24h", ep.filter("readings_before_failure > 0").count(), 387)

### 6c. Detect missing telemetry intervals – `silver_telemetry_gaps`
A gap = no reading from a machine for more than 24 hours.

In [0]:
GAP_HOURS = 24

def find_gaps(df, gap_hours):
    w = Window.partitionBy("machine_id").orderBy("reading_ts")
    return (df.select("machine_id", "reading_ts")
              .withColumn("gap_start", F.lag("reading_ts").over(w))
              .withColumn("gap_hours", F.round((F.unix_timestamp("reading_ts") - F.unix_timestamp("gap_start")) / 3600, 1))
              .filter(F.col("gap_hours") > gap_hours)
              .select("machine_id", "gap_start", F.col("reading_ts").alias("gap_end"), "gap_hours"))

save(find_gaps(spark.table("silver_sensor_readings"), GAP_HOURS), "silver_telemetry_gaps")

g = spark.table("silver_telemetry_gaps")
check("T6", "telemetry gaps > 24h",        g.count(), 2836)
check("T6", "all gaps longer than 24h",    g.filter(f"gap_hours <= {GAP_HOURS}").count(), 0)
display(g.orderBy(F.desc("gap_hours")).limit(10))

## 7. Unit tests
Small hand-made rows run through the **same functions** used above (`type_sensor`, `dedup`, `apply_sensor_rules`, `find_gaps`).
Late / out-of-order / replay CDC cases are tested in section 5.

In [0]:
cols = "reading_id string, machine_id string, plant_id string, reading_ts string, temperature_c string, vibration_mm_s string, " \
       "pressure_bar string, rpm string, operating_mode string, sensor_quality string, firmware_version string, received_ts string, " \
       "_source_file string, _ingestion_ts timestamp"
m1 = spark.table("silver_machine_master").first()["machine_id"]     # a real machine id
t0 = spark.sql("SELECT current_timestamp()").first()[0]

def row(rid, mid, rts, temp, file="f1.csv", recv="2026-07-01T00:40:00Z"):
    return (rid, mid, "P1", rts, temp, "1.0", "5.0", "1000", "RUNNING", "GOOD", "v1.0", recv, file, t0)

test_rows = [
    row("T_VALID",  m1,     "2026-07-01T00:00:00Z", "50.0"),                       # valid
    row("T_DUP",    m1,     "2026-07-01T01:00:00Z", "50.0", file="a.csv"),         # duplicate - first copy
    row("T_DUP",    m1,     "2026-07-01T01:00:00Z", "50.0", file="b.csv"),         # duplicate - second copy
    row(None,       m1,     "2026-07-01T02:00:00Z", "50.0"),                       # null key
    row("T_BADTS",  m1,     "not-a-date",           "50.0"),                       # invalid timestamp
    row("T_NOMACH", "MX99", "2026-07-01T03:00:00Z", "50.0"),                       # unknown machine
    row("T_TEMP",   m1,     "2026-07-01T04:00:00Z", "999999.99"),                  # sentinel temperature
    row("T_LATE",   m1,     "2026-07-03T00:00:00Z", "50.0", recv="2026-07-05T00:00:00Z"),  # late arrival, still valid
]
result = {(r["reading_id"], r["_source_file"]): r["rule_id"]
          for r in apply_sensor_rules(dedup(type_sensor(spark.createDataFrame(test_rows, cols)))).collect()}

check("T7", "valid row passes",                 result[("T_VALID", "f1.csv")],  None)
check("T7", "duplicate: first copy kept",       result[("T_DUP", "a.csv")],     None)
check("T7", "duplicate: second copy DQ05",      result[("T_DUP", "b.csv")],     "DQ05")
check("T7", "null reading_id -> DQ01",          result[(None, "f1.csv")],       "DQ01")
check("T7", "invalid timestamp -> DQ02",        result[("T_BADTS", "f1.csv")],  "DQ02")
check("T7", "unknown machine -> DQ03",          result[("T_NOMACH", "f1.csv")], "DQ03")
check("T7", "temperature sentinel -> DQ04",     result[("T_TEMP", "f1.csv")],   "DQ04")
check("T7", "late-arriving reading still valid", result[("T_LATE", "f1.csv")],  None)

gap_test = spark.createDataFrame([("M1", "2026-07-01T00:00:00Z"), ("M1", "2026-07-01T10:00:00Z"), ("M1", "2026-07-03T10:00:00Z")],
                                 "machine_id string, reading_ts string").withColumn("reading_ts", F.to_timestamp("reading_ts"))
check("T7", "gap detection finds the 48h gap only", [r["gap_hours"] for r in find_gaps(gap_test, 24).collect()], [48.0])

## 8. Final validation summary

In [0]:
summary = spark.createDataFrame(results, "task string, check_name string, actual string, expected string, status string")
display(summary)
display(summary.groupBy("status").count())

failed = summary.filter("status = 'FAIL'").count()
assert failed == 0, f"{failed} check(s) FAILED - see table above"
print(f"Day 2 complete - all {summary.count()} checks PASSED")

In [0]:
display(spark.sql(f"SHOW TABLES IN {SILVER}"))